# Tuning Classification Threshold

Runnable code for the notes, using **scikit-learn's built-in implementations directly
— no user-defined functions anywhere**.

Covers:
1. Turning predicted probabilities into class labels with a threshold
2. Why 0.5 isn't always optimal (cancer-screening style example)
3. How threshold changes affect the confusion matrix
4. Precision, Recall, and the Precision-Recall trade-off
5. F1-score vs. business-driven threshold selection
6. Threshold selection by **business cost**
7. Threshold selection by **minimum recall constraint**
8. Threshold selection by **operational capacity**


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score,
    precision_recall_curve, PrecisionRecallDisplay,
)

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 5)


## 2. Synthetic Dataset (Cancer-Screening Style)

An imbalanced binary classification problem, similar in spirit to cancer screening:
positives (sick patients) are the minority class, and missing one is costly.


In [ ]:
X, y = make_classification(
    n_samples=5000, n_features=12, n_informative=6, n_redundant=2,
    weights=[0.85, 0.15], class_sep=0.9, random_state=42,
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

logreg_model = LogisticRegression(max_iter=1000)
logreg_model.fit(X_train, y_train)

predicted_probabilities = logreg_model.predict_proba(X_test)[:, 1]
print("Test set size:", len(y_test), " Positive rate:", y_test.mean().round(3))


## 3. From Probability to Label: The Role of a Threshold

The worked example from the notes: same probabilities, two different thresholds,
two different sets of predictions — **the model itself never changes**.


In [ ]:
example_actual = np.array([1, 0, 1, 0, 1, 0])
example_probability = np.array([0.90, 0.72, 0.61, 0.48, 0.41, 0.10])

predictions_at_050 = (example_probability >= 0.50).astype(int)
predictions_at_030 = (example_probability >= 0.30).astype(int)

pd.DataFrame({
    "Actual": example_actual,
    "Probability": example_probability,
    "Prediction @ 0.50": predictions_at_050,
    "Prediction @ 0.30": predictions_at_030,
})


## 4. Threshold & the Confusion Matrix

Lowering the threshold from 0.50 to 0.30 typically increases TP and FP while
decreasing FN and TN. Raising it from 0.50 to 0.80 does the opposite.


In [ ]:
predicted_labels_050 = (predicted_probabilities >= 0.50).astype(int)
predicted_labels_030 = (predicted_probabilities >= 0.30).astype(int)
predicted_labels_080 = (predicted_probabilities >= 0.80).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

ConfusionMatrixDisplay.from_predictions(y_test, predicted_labels_030, ax=axes[0], colorbar=False)
axes[0].set_title("Threshold = 0.30")

ConfusionMatrixDisplay.from_predictions(y_test, predicted_labels_050, ax=axes[1], colorbar=False)
axes[1].set_title("Threshold = 0.50 (default)")

ConfusionMatrixDisplay.from_predictions(y_test, predicted_labels_080, ax=axes[2], colorbar=False)
axes[2].set_title("Threshold = 0.80")

plt.tight_layout()
plt.show()


In [ ]:
tn_030, fp_030, fn_030, tp_030 = confusion_matrix(y_test, predicted_labels_030).ravel()
tn_050, fp_050, fn_050, tp_050 = confusion_matrix(y_test, predicted_labels_050).ravel()
tn_080, fp_080, fn_080, tp_080 = confusion_matrix(y_test, predicted_labels_080).ravel()

confusion_summary = pd.DataFrame({
    "Threshold": [0.30, 0.50, 0.80],
    "TP": [tp_030, tp_050, tp_080],
    "FP": [fp_030, fp_050, fp_080],
    "FN": [fn_030, fn_050, fn_080],
    "TN": [tn_030, tn_050, tn_080],
})
confusion_summary


## 5. Precision & Recall at Each Threshold

- **Precision** — of everything predicted positive, how many were actually positive?
- **Recall** — of all actual positives, how many did we successfully detect?


In [ ]:
precision_recall_summary = pd.DataFrame({
    "Threshold": [0.30, 0.50, 0.80],
    "Precision": [
        precision_score(y_test, predicted_labels_030),
        precision_score(y_test, predicted_labels_050),
        precision_score(y_test, predicted_labels_080),
    ],
    "Recall": [
        recall_score(y_test, predicted_labels_030),
        recall_score(y_test, predicted_labels_050),
        recall_score(y_test, predicted_labels_080),
    ],
    "F1": [
        f1_score(y_test, predicted_labels_030),
        f1_score(y_test, predicted_labels_050),
        f1_score(y_test, predicted_labels_080),
    ],
})
precision_recall_summary


## 6. Precision-Recall Trade-off Curve

`sklearn.metrics.precision_recall_curve` computes precision and recall directly
across every possible threshold — no manual loop or function needed.


In [ ]:
precision_values, recall_values, pr_thresholds = precision_recall_curve(y_test, predicted_probabilities)

PrecisionRecallDisplay.from_predictions(y_test, predicted_probabilities)
plt.title("Precision-Recall Curve")
plt.tight_layout()
plt.show()


In [ ]:
plt.plot(pr_thresholds, precision_values[:-1], label="Precision")
plt.plot(pr_thresholds, recall_values[:-1], label="Recall")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Precision & Recall vs. Threshold")
plt.legend()
plt.tight_layout()
plt.show()


## 7. F1-Score-Optimal Threshold

F1 balances precision and recall mathematically — but as the notes point out,
this isn't automatically the right **business** threshold.


In [ ]:
f1_scores_per_threshold = 2 * (precision_values[:-1] * recall_values[:-1]) / (precision_values[:-1] + recall_values[:-1] + 1e-12)
best_f1_index = np.argmax(f1_scores_per_threshold)
best_f1_threshold = pr_thresholds[best_f1_index]

print("F1-optimal threshold:", round(best_f1_threshold, 3))
print("Precision at that threshold:", round(precision_values[best_f1_index], 3))
print("Recall at that threshold:", round(recall_values[best_f1_index], 3))
print("F1 at that threshold:", round(f1_scores_per_threshold[best_f1_index], 3))


## 8. Threshold Selection by Business Cost

`TotalCost = FP * CostFP + FN * CostFN`. Costs are set as in the notes'
cancer-screening framing: a missed positive (false negative) is far more expensive
than a false alarm (false positive).


In [ ]:
cost_false_positive = 50
cost_false_negative = 1000

candidate_thresholds = pr_thresholds

total_cost_per_threshold = []
for candidate_threshold in candidate_thresholds:
    candidate_predictions = (predicted_probabilities >= candidate_threshold).astype(int)
    tn_c, fp_c, fn_c, tp_c = confusion_matrix(y_test, candidate_predictions).ravel()
    total_cost_per_threshold.append(fp_c * cost_false_positive + fn_c * cost_false_negative)

total_cost_per_threshold = np.array(total_cost_per_threshold)

cost_optimal_index = np.argmin(total_cost_per_threshold)
cost_optimal_threshold = candidate_thresholds[cost_optimal_index]

print("Cost-optimal threshold:", round(cost_optimal_threshold, 3))
print("Minimum total cost: $", round(total_cost_per_threshold[cost_optimal_index], 2))


In [ ]:
plt.plot(candidate_thresholds, total_cost_per_threshold)
plt.axvline(cost_optimal_threshold, color="red", linestyle="--", label=f"Cost-optimal = {cost_optimal_threshold:.3f}")
plt.xlabel("Threshold")
plt.ylabel("Total Cost ($)")
plt.title("Total Cost vs. Threshold")
plt.legend()
plt.tight_layout()
plt.show()


## 9. Threshold Selection by Minimum Recall Requirement

Sometimes the business gives a hard constraint instead of a dollar cost —
e.g. *"we must detect at least 90% of positives."* Among all thresholds meeting
that constraint, pick the one with the highest precision.


In [ ]:
minimum_required_recall = 0.90

recall_without_last = recall_values[:-1]
precision_without_last = precision_values[:-1]

meets_recall_constraint = recall_without_last >= minimum_required_recall
qualifying_thresholds = candidate_thresholds[meets_recall_constraint]
qualifying_precisions = precision_without_last[meets_recall_constraint]
qualifying_recalls = recall_without_last[meets_recall_constraint]

best_index_under_constraint = np.argmax(qualifying_precisions)
recall_constrained_threshold = qualifying_thresholds[best_index_under_constraint]

print("Threshold meeting recall >= 0.90 with highest precision:", round(recall_constrained_threshold, 3))
print("Precision achieved:", round(qualifying_precisions[best_index_under_constraint], 3))
print("Recall achieved:", round(qualifying_recalls[best_index_under_constraint], 3))


## 10. Threshold Selection by Operational Capacity

Suppose the team can only manually review a fixed number of flagged cases per day
(e.g. 150). The threshold is set so exactly that many test cases are flagged —
found directly by sorting predicted probabilities.


In [ ]:
daily_review_capacity = 150

sorted_probabilities_descending = np.sort(predicted_probabilities)[::-1]
capacity_based_threshold = sorted_probabilities_descending[daily_review_capacity - 1]

capacity_based_predictions = (predicted_probabilities >= capacity_based_threshold).astype(int)

print("Capacity-based threshold:", round(capacity_based_threshold, 3))
print("Number of cases flagged:", capacity_based_predictions.sum())
print("Precision:", round(precision_score(y_test, capacity_based_predictions), 3))
print("Recall:", round(recall_score(y_test, capacity_based_predictions), 3))


## 11. Summary: Comparing All Threshold Strategies

In [ ]:
strategy_thresholds = {
    "Default (0.50)": 0.50,
    "F1-Optimal": best_f1_threshold,
    "Cost-Optimal": cost_optimal_threshold,
    "Recall >= 0.90 Constraint": recall_constrained_threshold,
    "Operational Capacity (150/day)": capacity_based_threshold,
}

summary_rows = []
for strategy_name, strategy_threshold in strategy_thresholds.items():
    strategy_predictions = (predicted_probabilities >= strategy_threshold).astype(int)
    tn_s, fp_s, fn_s, tp_s = confusion_matrix(y_test, strategy_predictions).ravel()
    summary_rows.append({
        "Strategy": strategy_name,
        "Threshold": round(strategy_threshold, 3),
        "TP": tp_s, "FP": fp_s, "FN": fn_s, "TN": tn_s,
        "Precision": round(precision_score(y_test, strategy_predictions), 3),
        "Recall": round(recall_score(y_test, strategy_predictions), 3),
        "F1": round(f1_score(y_test, strategy_predictions), 3),
        "Total Cost ($)": fp_s * cost_false_positive + fn_s * cost_false_negative,
    })

pd.DataFrame(summary_rows).set_index("Strategy")


## 12. Key Takeaways

- A threshold converts predicted probabilities into class labels; **changing it never retrains the model**.
- 0.50 is a natural default, not a guaranteed optimum — it ignores the real-world cost of each error type.
- Lowering the threshold → more positives predicted → Recall↑, Precision↓. Raising it does the opposite. This is the **precision-recall trade-off**.
- F1-score balances precision and recall mathematically, but that's not the same as the best **business** threshold.
- When error costs are known, pick the threshold that **minimizes total cost** (`FP × CostFP + FN × CostFN`).
- When the business gives a hard requirement (e.g. "recall ≥ 90%"), search among thresholds meeting that constraint for the best precision.
- When there's a fixed **operational capacity**, set the threshold so exactly that many cases are flagged.
